# Realistic Vision V5.1 - Uncensored Studio (Tested & Verified)
Uses public non-gated Realistic Vision V5.1 safetensors checkpoint with zero download errors.

In [ ]:
!pip install -q diffusers transformers accelerate torch torchvision gradio safetensors

import torch
from diffusers import StableDiffusionImg2ImgPipeline
import gradio as gr
import os
import subprocess

has_cuda = torch.cuda.is_available()
device = "cuda" if has_cuda else "cpu"
dtype = torch.float16 if has_cuda else torch.float32

model_file = "realisticVisionV51.safetensors"
if not os.path.exists(model_file) or os.path.getsize(model_file) < 1000000:
    print("Downloading Realistic Vision V5.1 weights...")
    subprocess.run([
        "wget", "-O", model_file,
        "https://huggingface.co/SG161222/Realistic_Vision_V5.1_noVAE/resolve/main/Realistic_Vision_V5.1.safetensors"
    ])

print(
    "Loading Realistic Vision V5.1 into pipeline..."
)
pipe_i2i = StableDiffusionImg2ImgPipeline.from_single_file(
    model_file,
    torch_dtype=dtype,
    use_safetensors=True
).to(device)

if hasattr(pipe_i2i, "safety_checker"):
    pipe_i2i.safety_checker = None

def edit(image, prompt, negative_prompt, strength, steps, cfg):
    if image is None:
        return None
    
    image = image.convert(
        "RGB"
    )
    image.thumbnail((512, 512))
    
    full_prompt = prompt + ", raw photo, highly detailed, photorealistic"
    full_neg = negative_prompt + ", blurry, low quality, deformed"
    
    img = pipe_i2i(
        prompt=full_prompt,
        negative_prompt=full_neg,
        image=image,
        strength=float(strength),
        num_inference_steps=int(steps),
        guidance_scale=float(cfg)
    ).images[0]
    return img

demo = gr.Interface(
    fn=edit,
    inputs=[
        gr.Image(type="pil", label="Source Image"),
        gr.Textbox(label="Prompt", value="photorealistic, masterpiece, high quality"),
        gr.Textbox(label="Negative Prompt", value="blurry, low quality, deformed"),
        gr.Slider(0.1, 0.6, value=0.25, label="Strength (0.2 - 0.3 preserves identity)"),
        gr.Slider(20, 50, value=30, step=1, label="Steps"),
        gr.Slider(4.0, 10.0, value=7.0, step=0.5, label="CFG Scale")
    ],
    outputs=gr.Image(label="Result (Uncensored)"),
    title="Realistic Vision V5.1 Uncensored Studio"
)
demo.launch(share=True, debug=True)